# Retail Product Demand Estimation and Stockout Risk Analysis
## End-to-End Machine Learning Pipeline & Statistical Analysis

**Objectives:**
1. Clean and preprocess transaction records from the UCI Online Retail dataset.
2. Perform comprehensive exploratory data analysis (EDA).
3. Engineer time-series lag and rolling momentum features without data leakage.
4. Train and compare Naive/Seasonal Naive baselines alongside Ridge Regression, Random Forest, and Gradient Boosting.
5. Evaluate models using leakage-free chronological splits and encoding ablation.
6. Execute hyperparameter optimization using date-grouped expanding-window cross-validation and multi-origin recursive backtesting.
7. Implement explicit forecast-based inventory stockout risk rules.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

print('Libraries imported successfully.')

### 1. Load Cleaned Dataset and Validate Dimensions

In [2]:
cleaned_csv = '../dataset/cleaned_retail_data.csv'
df = pd.read_csv(cleaned_csv)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Cleaned records:', len(df))
print('Unique products:', df['StockCode'].nunique())
print('Date span:', df['InvoiceDate'].min(), 'to', df['InvoiceDate'].max())
df.head()

### 2. Feature Matrix and Time-Series Properties

In [3]:
featured_csv = '../models/featured_demand.csv.gz'
featured_df = pd.read_csv(featured_csv)
featured_df['Date'] = pd.to_datetime(featured_df['Date'])
print('Feature matrix shape:', featured_df.shape)
print('Engineered features:', list(featured_df.columns))
featured_df[['StockCode', 'Date', 'DailyDemand', 'lag_1', 'lag_7', 'rolling_mean_7', 'rolling_trend']].head()

### 3. Chronological Model Evaluation Comparison

In [4]:
metrics_file = '../outputs/model_metrics.json'
with open(metrics_file, 'r') as f:
    metrics = json.load(f)

metrics_df = pd.DataFrame(metrics['metrics']).T
print('Selected Champion (by CV RMSE):', metrics['best_model'])
print('Selected Encoding:', metrics.get('encoding_strategy'))
print('Horizon Backtesting (ML vs Seasonal Naive):', json.dumps(metrics.get('backtesting', {}).get('horizon_metrics', {}), indent=2))
metrics_df

### 4. Forecast-Driven Stockout Risk Simulation

In [5]:
import sys
sys.path.insert(0, '..')
from backend.stockout import analyze_stockout_risk

# Scenario 1: Low inventory, High Stockout Risk
risk_high = analyze_stockout_risk('85123A', current_inventory=50, supplier_lead_time=10)
print('Scenario 1 Status:', risk_high['stockout_status'])
print('Lead-time demand:', risk_high['expected_lead_time_demand'], 'units')
print('Suggested replenishment:', risk_high['suggested_replenishment'], 'units')

# Scenario 2: Healthy inventory, Low Stockout Risk
risk_low = analyze_stockout_risk('85123A', current_inventory=2500, supplier_lead_time=10)
print('\nScenario 2 Status:', risk_low['stockout_status'])
print('Remaining stock after lead time:', risk_low['inventory_remaining_after_lead_time'], 'units')